In [ ]:
#| hide
from nbdev.showdoc import *
from nbdev import show_doc
from fhemb.dbms.db import Datasource
from fhemb.utils.factories import ConcreteEmbFactory, ConcretePcaFactory

DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.paths
DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.db


# Datasource
> `Datasource` is the core container for thermal‑face features, lazy‑loaded embeddings, PCA projections, and correlation analysis.

The `Datasource` class (implemented as `BaseDatasource`) provides a unified interface for
working with per‑subject thermal‑face data. It manages raw heatmaps, derives histogram
and percentile features on demand, and exposes a consistent API for PCA, wavelet
decomposition, and lagged‑correlation analysis.

#### The Method Resolution Order (MRO) 
>the exact linear path Python follows when looking up attributes or when super() is used.

```raw
Datasource
 └── BaseDatasource
      └── ValidateSubjsMixin
           └── object
```

#### Architectural responsibilities

- **Derivation layer**: consume DB‑loaded lazy attributes provided by `Piece` and perform deterministic low‑level derivations (shared bin edges, histograms/bin heights, percentiles, basic statistical feature vectors).

- **Lazy‑attribute producer**: compute and cache derived lazy attributes (`binheights_ts`, `binedges`, `percentiles_ts`, `features_ts`) that the rest of the system uses to build embeddings.

- **Time alignment and provenance**: enforce consistent preprocessing rules (thresholding, alignment) so embeddings constructed from these matrices are reproducible and auditable.

- **Key relationship**: `Datasource` derives its lazy attributes from the DB lazy attributes that `Piece` loads — this DB → derived separation is a central architectural feature.

In [ ]:
show_doc(Datasource, name="The Constructor", title_level=2)

---

## The Constructor

```python

def Datasource(
    time_interval:Tuple[int, int], # Time interval for the datasource (start, end).
    features_tint:Dict[Subjs, List[np.matrix]], # matrix features x time
    subjects:List[str]=None, # List of subjects to include in the datasource. If None, all subjects from features_tint will be used.
    piece_attributes:Dict[str, Union[str, int]]=None, # Dictionary of attributes related to the piece (e.g., 'piece_name', 'composer', 'performance_date').
    position:Dict[str, List[np.ndarray]]=None, # Dictionary mapping subjects to their face positions over time (faceposition_t).
):


```

*Initialize the Datasource with time interval, features, subjects, and piece attributes.*

## Main Methods

In [ ]:
show_doc(Datasource.project, name='project', title_level=3)

---

### project

```python

def project(
    subjs:Union[List[int], str]='all', # List of subject indices or `'all'`.
    features:Optional[List[str]]=None, # Feature names to project.
    heatmap:Optional[np.ndarray]=None, # Heatmap data to project.
    fvector:Optional[np.ndarray]=None, # Feature vector to project.
    fbands:Optional[List[int]]=None, # Frequency bands to project.
    pcfactory:Optional['ConcretePcaFactory']=None, # PCA factory configured with the desired scaler and PCA model.
    wdfactory:Optional['ConcreteEmbFactory']=None, # Wavelet factory for spectral decomposition.
)->Embedding: # The projected embedding.


```

*Project selected features, heatmaps, or feature vectors for the specified subjects*
and frequency bands.

In [ ]:

show_doc(Datasource.create_pca_embedding, name='create_pca_embedding', title_level=3)

---

### create_pca_embedding

```python

def create_pca_embedding(
    factory:'ConcretePcaFactory', # Factory that creates the PCA dictionary.
)->Tuple['Embedding', Tuple]: # The created PCA embedding.


```

*Create a PCA embedding using the provided factory.*

In [ ]:
#| hide
from fhemb.utils.factories import pca_factory

The `ConcretePcaFactory` is defined by:

In [ ]:
show_doc(pca_factory, title_level=4)

---

#### pca_factory

```python

def pca_factory(
    name:NoneType=None, # The name of the factory. Default is None. Is used for saving/loading PCA models.
    scaler:str='Standard', # The type of scaler to use. Default is "Standard". Options are "Standard", "Robust".
    pca:str='Kernel', # The type of PCA to use. Default is "Kernel". Options are "Kernel", "PCA", "Sparse", "IncrementalPCA".
    scaler_kwargs:NoneType=None, # Additional keyword arguments to pass to the scaler constructor. Default is None.
    pca_kwargs:NoneType=None, # Additional keyword arguments to pass to the PCA constructor. Default is None.
)->ConcretePcaFactory: # An instance of ConcretePcaFactory configured with the specified scaler and PCA.


```

*Factory function to create a ConcretePcaFactory with specified scaler and PCA types.*

In [ ]:
show_doc(ConcretePcaFactory, name="The constructor", title_level=5)

---

##### The constructor

```python

def ConcretePcaFactory(
    scaler:Callable, pca:Callable, name:NoneType=None, # Optional name used for saving/loading PCA models.
):


```

*Factory for PCA pipelines and eigenface embeddings.*

This factory builds PCA pipelines from provided scaler and PCA constructors,
fits them on per-subject data, and produces embeddings and reconstructed
eigenfaces.

## Lag / Correlation Analysis

In [ ]:
show_doc(Datasource.lagged_correlations_df, name='lagged_correlations_df', title_level=4)

---

#### lagged_correlations_df

```python

def lagged_correlations_df(
    subjs, # Subjects to analyze.
    features, # Feature names or identifiers.
    fbands, # Frequency bands.
    twin, # Time window in frames.
    max_lag, # Maximum lag in frames.
    step, # Step size in frames.
    wdfactory:NoneType=None, # Wavelet factory for spectral decomposition.
    pcfactory:NoneType=None, # PCA factory.
    bbootstrap:bool=False, # Whether to include block bootstrap resampling.
    n_jobs:int=4, # Number of parallel jobs.
): # Dictionary with keys 'df' and 'df_bbootstrap' (if bbootstrap is True).


```

*Calculate lagged correlations between two features of a subject or between two subjects on the same feature.*

In [ ]:
show_doc(Datasource.plot_lagged_correlations, name='plot_lagged_correlations', title_level=4)

---

#### plot_lagged_correlations

```python

def plot_lagged_correlations(
    subjs, # Subjects to analyze.
    features, # Feature names or identifiers.
    fbands:list=[], # Frequency bands (default: use original signal).
    twin:int=50, # Time window in frames.
    max_lag:int=200, # Maximum lag in frames.
    step:int=25, # Step size in frames.
    wdfactory:NoneType=None, # Wavelet factory for spectral decomposition.
    pcfactory:NoneType=None, # PCA factory.
    bbootstrap:bool=False, # Whether to include block bootstrap resampling.
    n_jobs:int=4, # Number of parallel jobs.
    kwargs:VAR_KEYWORD
):


```

*Analyze and visualize rolling windowed time-lagged cross-correlations between subjects or features.*

In [ ]:
show_doc(Datasource.lagged_correlation_graph, name='lagged_correlation_graph', title_level=4)

---

#### lagged_correlation_graph

```python

def lagged_correlation_graph(
    subjs:list=[], # List of subjects.
    features:list=[], # List of features.
    lag:int=0, # Lag to plot.
    fbands:list=[], # Frequency bands.
    factory:NoneType=None, # Wavelet factory object.
    twin:int=50, # Time window size.
    step:int=25, # Step size.
    max_lag:int=200, # Maximum lag.
    bbootstrap:bool=False, # Whether to include block bootstrap resampling.
    n_jobs:int=4, # Number of jobs.
):


```

*Plot lagged correlations between features for each subject.*

## Distribution / Summary

In [ ]:
show_doc(Datasource.plot_pdf, name='plot_pdf', title_level=4)

---

#### plot_pdf

```python

def plot_pdf(
    subj, # Subject index.
    secs, # List of seconds to plot.
    bw_method:str | float=1, # Bandwidth method for kernel density estimation ('scott', 'silverman', or a scalar).
):


```

*Plot the probability density function (PDF) at specified seconds for a subject.*

## DatasourceResized
> This is a normalized version of `Datasource`. In contrast to `Datasource`, the `DatasourceResized` makes sure that the shapes of arrays across time are fixed. This is required for example for PCA.   

## Properties *(readonly)*

### Lazy loaded properties
>The lazy properties are derived on demand from the raw heatmaps and then cached.

#### `Datasource` property of type `DatasourceResized`

- **`rsfacebbox_t`**

::: {.callout-important}
`DatasourceResized` is a resized version of `BaseDatasource`. 

```raw
DatasourceResized
 └── BaseDatasource
      └── ValidateSubjsMixin
           └── object
```

In contrast to `Datasource`, the `DatasourceResized` 

- makes sure that the shapes of arrays across time are fixed. This is required for example in PCA.

- has two additional lazy loaded properties of type `Embedding`:

  - **`normfacevec_ts`** is a time series of raw facial heatmaps.
  
  - **`bboxheatmap_ts`** is a time series of grids of temperature means: each mean is calculated for a sub-ROI in a (resized) face bounding boxe.
:::



#### `Datasource` and `Datasourceresized` properties of type `Embeddiong`

- **`features_ts`** 

- **`percentiles_ts`**   

- **`binheights_ts`**   

---

### Other useful properties

- **`position`** *(`Embedding`)*

- **`binedges`**  

- **`features_tint`** 

- **`facesizes_t`**   

- **`sr = 25`** *(sampling rate)*